# Solar filament instance segmentation: end-to-end pipeline

IEEE BigData Cup 2026, Challenge 02 (filament segmentation in GONG H-alpha full-disk images). This notebook
reproduces the submitted model **final-A** from its released weights. It also contains every other step of
the project: the judging metrics, the analyses and the complete training chain.

| § | Step | Default | Hardware, time |
|---|---|---|---|
| 0 | settings | runs | |
| 1 | installation: pinned requirements, package, official DCNv4 CUDA operator | runs | ~5 min |
| 2 | data check; the model's input channels | runs | CPU, seconds |
| 3 | trained weights: optional download, SHA-256, checkpoint metadata | runs | CPU |
| 4 | unit tests | runs | CPU, ~2 min |
| 5 | **test inference**: 8 dihedral views → fusion → submission CSV → checks | runs | 1 GPU: ~11 min on an RTX 3090, then ~1 min CPU |
| 6 | morphology of the test predictions | runs | CPU |
| 7 | judging metrics on the held-out fold 0 | shows the released results; `RUN_FOLD0` recomputes them | GPU ~8 min + CPU |
| 8 | label visibility and magnetogram analysis | shows the released results; `RUN_RESEARCH` recomputes them | CPU |
| 9 | training chain from the ImageNet backbone, with Noisy-Student self-training | off (`RUN_TRAINING`) | several GPU-days |

**Results.**
- Public leaderboard: **PQ 0.40** (single model, 8-view test-time augmentation).
- Held-out fold 0, using the fold-0 twin of the final recipe: PQ 0.446, mean Dice of overlapping pairs 0.728.
- Two human annotators of the same image agree at PQ 0.340.

Every code cell calls the repository's scripts with the same commands as the README, so the notebook and the
command line give the same results. The method is in `docs/TECHNICAL_REPORT.md`, and the 4-page paper is in `paper/`.

**Compliance.**
- Inference reads only the provided H-alpha test image (rule 1).
- Training uses the competition polygons and our own pseudo-labels of unlabelled GONG archive frames. It uses no annotator codes or other ground-truth metadata, and no archive frame from a test-image day (rule 2).
- The magnetograms of §8 are used for analysis only.

## 0. Settings

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, subprocess, sys, time

REPO = Path.cwd() if (Path.cwd() / "solar_filaments").is_dir() else Path.cwd().parent
os.chdir(REPO)
sys.path.insert(0, str(REPO))

DATA_ROOT = REPO / "data/raw/MAGFiLO_1.0_Kaggle_2026"   # competition files: train/, test/test_images/
WEIGHTS = REPO / "weights/final_a.pt"                    # the submitted model
WEIGHTS_URL = "https://github.com/ukiyois/solar/releases/download/v1.0.0/final_a.pt"  # used only if WEIGHTS is missing
WEIGHTS_SHA256 = "6784bcc33d38f6495aea322863d83ccd062f7297686c0b9d5edcbc89f48933cb"
SUBMITTED = REPO / "deliverables/submission_single_ffinal_a_m40.csv"  # the scored file, if available
OUT = REPO / "runs/notebook"                             # everything this notebook writes

INSTALL = True              # pinned requirements + the package (installs PyTorch 2.10 + CUDA 12.8 if missing)
BUILD_DCNV4 = True          # official DCNv4 CUDA operator (needs nvcc); otherwise the slower PyTorch reference
RUN_TESTS = True            # unit tests (CPU)
RUN_INFERENCE = True        # test-set inference and submission (needs a CUDA GPU)
RUN_FOLD0 = False           # recompute the fold-0 judging metrics with a model trained without fold-0 labels:
FOLD0_CHECKPOINT = REPO / "runs/v9_e9/best.pt"           #   the fold-0 twin of the final recipe (E9, section 9)
RUN_RESEARCH = False        # recompute the label-visibility / magnetogram analyses (needs the fold-0 dump)
RUN_TRAINING = False        # the complete training chain (several GPU-days)
GPUS = 1                    # GPUs per training stage (torch.distributed); the batch size is per GPU
WORKERS = 4                 # CPU processes for fusion and metrics

VIEWS = "d0,d1,d2,d3,f0,f1,f2,f3"   # identity, 3 rotations, and the 4 flipped versions
TEST_DUMP, FOLD0_DUMP = OUT / "tta/final_a/test", OUT / "tta/fold0/val"
SUBMISSION = OUT / "submission.csv"
OUT.mkdir(parents=True, exist_ok=True)


def run(*args):
    """Run one repository command, stream its output into the notebook, raise if it fails."""
    command = [str(a) for a in args]
    print("$", " ".join(command), flush=True)
    start = time.time()
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)
    print(f"[done in {time.time() - start:.0f} s]")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


print("repository:", REPO)
print("python", platform.python_version(), "|", platform.platform())

## 1. Installation

Use Linux with an NVIDIA GPU and a CUDA 12.8 toolkit (`nvcc`), which the official DCNv4 kernel needs. The
project was verified with Python 3.10. `requirements.txt` pins every package the code uses, PyTorch 2.10.0
(CUDA 12.8 wheels) included. Without the compiled operator the model falls back to a PyTorch reference
implementation of DCNv4, which is slower.

In [ ]:
if INSTALL:
    run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")   # all packages, pinned
    run(sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-build-isolation")
if BUILD_DCNV4:
    try:
        run(sys.executable, "scripts/build_dcnv4.py")
    except subprocess.CalledProcessError:
        print("DCNv4 build failed: continuing with the PyTorch reference operator (slower)")

In [ ]:
import torch
from solar_filaments.ops.dcnv4 import native_function

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}: {properties.name}, {properties.total_memory / 2**30:.0f} GB")
print("native DCNv4 operator:", native_function() is not None)
if os.environ.get("CUDA_VISIBLE_DEVICES") == "":
    print("warning: CUDA_VISIBLE_DEVICES is set but empty, so every GPU is hidden")

## 2. Data and model inputs

The competition files are not redistributed. Place them as follows:

```text
data/raw/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
data/raw/MAGFiLO_1.0_Kaggle_2026/train/train_images/*.jpeg
data/raw/MAGFiLO_1.0_Kaggle_2026/test/test_images/*.jpeg
```

Expected: 707 training observations with 1,154 annotator records and 8,199 filament polygons, and 180 test
observations. One observation is often drawn by two or three annotators. The metric scores a prediction
against each annotator record separately.

In [ ]:
annotations = json.loads((DATA_ROOT / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json").read_text(encoding="utf-8"))
observations = {record["file_name"] for record in annotations["images"]}
test_images = sorted(p for p in (DATA_ROOT / "test/test_images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg"})
print(f"training: {len(observations)} observations, {len(annotations['images'])} annotator records, "
      f"{len(annotations['annotations'])} filament polygons")
print(f"test: {len(test_images)} observations")
assert len(test_images) == 180, "expected the 180 competition test images"

**What the model sees.** Every input is computed from the test image itself (`solar_filaments/model.py`):

- **Context view.** The full disk is resized to 1024 px for the pretrained FlashInternImage-L backbone.
  Filaments are hundreds of pixels long, and the backbone needs the whole disk to tell them from sunspots,
  plage and the limb.
- **Native channels at 2048 px.** These feed the detail path:
  - the intensity $I$;
  - the local contrast $4\,(I - \overline{I}_{15\times15})$;
  - the **limb-flattened relative darkness** $4\,(I/B - 1)$. Filaments are absorption features. $B$ is the
    quiet-Sun background, estimated on the disk only by a normalized convolution (σ ≈ 32 px), so
    $I/B - 1$ is the cloud-model contrast. It removes the center-to-limb darkening that makes absolute
    intensity misleading near the limb.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.nn import functional as F
from solar_filaments.model import limb_flattened

example = test_images[0]
image = torch.from_numpy(np.asarray(Image.open(example).convert("L"), dtype=np.float32) / 255)[None, None]
channels = [("intensity $I$", image, (0, 1)),
            ("local contrast $4(I - \\overline{I}_{15})$", 4 * (image - F.avg_pool2d(image, 15, 1, 7)), (-.3, .3)),
            ("limb-flattened darkness $4(I/B - 1)$", limb_flattened(image), (-1, 1))]
fig, axes = plt.subplots(1, 3, figsize=(15, 5.4))
for ax, (title, channel, (low, high)) in zip(axes, channels):
    ax.imshow(channel[0, 0].numpy(), cmap="gray", vmin=low, vmax=high)
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(f"{example.name}: native model inputs")
plt.tight_layout()
plt.show()

## 3. Trained weights

`final_a.pt` stores the EMA weights of final-A together with its model configuration. The checksum below
identifies the submitted model. Expected: 939 tensors, EMA weights, 12 epochs (index 11),
initialized from F1 (`runs/v9_f1/final.pt`).

In [ ]:
if not WEIGHTS.exists() and WEIGHTS_URL:
    import requests
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    partial = WEIGHTS.with_suffix(".part")
    with requests.get(WEIGHTS_URL, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(partial, "wb") as stream:
            for block in response.iter_content(8 << 20):
                stream.write(block)
    partial.replace(WEIGHTS)
assert WEIGHTS.exists(), f"place final_a.pt at {WEIGHTS} or set WEIGHTS_URL (README, 'Reproduce the submission')"
assert sha256(WEIGHTS) == WEIGHTS_SHA256, "checksum mismatch: this is not the submitted model"
checkpoint = torch.load(WEIGHTS, map_location="cpu", weights_only=False, mmap=True)
config = checkpoint["model_config"]
print(f"{WEIGHTS.name}: SHA-256 OK | {len(checkpoint['model'])} tensors | EMA weights: {checkpoint['uses_ema']} | "
      f"epoch index {checkpoint['epoch']} | initialized from {checkpoint['initialized_from']}")
print({key: config[key] for key in ("backbone", "context_size", "queries", "decoder_layers", "seed_tokens",
                                    "detail_downsample", "detail_flattening", "curve_enabled")})
del checkpoint

## 4. Unit tests

The unit tests cover the following (2 of them need the compiled DCNv4 extension):
- DCNv4 packing and gradients;
- equivalence of tiled and untiled inference;
- losses on empty and mixed batches;
- PQ edge cases;
- isolation of the mask pilots;
- exactness of the warm start;
- fast evaluation against the reference implementation;
- checkpoint handling.

In [ ]:
if RUN_TESTS:
    run(sys.executable, "-m", "unittest", "discover", "-s", "tests")

## 5. Test inference and submission

1. **Eight dihedral views** (`scripts/dump_tta_candidates.py`, GPU). Each test image is segmented in its 4
   rotations by k·90° and their 4 flipped versions. These are exact pixel permutations, so thin filaments are
   never interpolated. For every view, each query with score P(filament)·σ(quality) ≥ 0.02 is stored as a
   probability crop, mapped back to the original orientation.
2. **Fusion and selection** (`scripts/fuse_tta.py`, CPU).
   - Candidates are clustered across views by IoU > 0.5, with at most one member per view.
   - Probabilities are averaged over the members, and the fused score is the summed member score / 8.
   - An instance is kept with ≥ 1 of 8 votes, score ≥ 0.10 and mask threshold 0.40.
   - Pixel ownership runs in score order: no overlaps, ≥ 32 px and ≥ 50 % of the area retained.
   - The thresholds were chosen on fold 0, not on the leaderboard.
3. **Checks** (`scripts/validate_submission.py`): the header, test ids, RLE decoding and non-overlap.

The submitted file has 1,072 instances on 179 of the 180 test images, no overlapping pixels, and areas of
253–23,490 px (median 1,385). A run on another GPU or driver can differ in a few pixels.

In [ ]:
if RUN_INFERENCE:
    assert torch.cuda.is_available(), "test inference needs a CUDA GPU"
    run(sys.executable, "-m", "scripts.dump_tta_candidates", "--checkpoint", WEIGHTS, "--split", "test", "--output", TEST_DUMP)
    run(sys.executable, "-m", "scripts.fuse_tta", "--folders", TEST_DUMP, "--votes", "0.125", "--scores", "0.10",
        "--masks", "0.40", "--test", "--workers", WORKERS, "--output", SUBMISSION)
elif not SUBMISSION.exists() and SUBMITTED.exists():
    SUBMISSION = SUBMITTED   # inspect the scored file instead
run(sys.executable, "scripts/validate_submission.py", SUBMISSION)

In [ ]:
import csv
from collections import Counter, defaultdict
from pycocotools import mask as mask_utils


def read_submission(path):
    by_image = defaultdict(list)
    with open(path, encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            by_image[row["filament_id"].rsplit("_", 1)[0]].append(
                {"size": [2048, 2048], "counts": row["segmentation_rle"].encode("ascii")})
    return by_image


predicted = read_submission(SUBMISSION)
counts = [len(predicted.get(p.stem, [])) for p in test_images]
areas = np.array([mask_utils.area(rle) for rles in predicted.values() for rle in rles])
print(f"{areas.size} instances on {sum(c > 0 for c in counts)} of {len(test_images)} test images; "
      f"per image median {np.median(counts):.0f}, max {max(counts)}; area median {np.median(areas):.0f} px "
      f"(range {areas.min()}-{areas.max()})")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].hist(counts, bins=np.arange(max(counts) + 2) - .5, color="tab:blue")
axes[0].set_xlabel("predicted filaments per test image")
axes[1].hist(np.log10(areas), bins=40, color="tab:orange")
axes[1].set_xlabel("log10 instance area (px)")
plt.tight_layout()
plt.show()

if SUBMITTED.exists() and SUBMISSION != SUBMITTED:   # agreement with the scored file
    scored = read_submission(SUBMITTED)
    matched, ious = 0, []
    for stem, rles in scored.items():
        mine = predicted.get(stem, [])
        if mine:
            iou = np.asarray(mask_utils.iou(mine, rles, [0] * len(rles))).reshape(len(mine), len(rles))
            matched += int((iou.max(0) > .5).sum())
            ious += iou.max(0)[iou.max(0) > .5].tolist()
    total = sum(map(len, scored.values()))
    print(f"scored file: {total} instances; reproduced at IoU > .5: {matched} (mean IoU {np.mean(ious):.4f})")

## 6. Morphology of the test predictions

The test observations with the most, a median number of and the fewest predicted instances. In each figure,
the top row shows the full disk and the bottom row a 512 px window around the largest instance. Instances keep
the shape of the filament spine, its curvature and its barbs at native resolution.

In [ ]:
from IPython.display import Image as Picture, Markdown, display

run(sys.executable, "-m", "scripts.morphology_figures", "--csv", SUBMISSION, "--label", "final-A",
    "--output-dir", OUT / "morphology_test")
for png in sorted((OUT / "morphology_test").glob("*.png")):
    display(Picture(filename=str(png), width=900))

## 7. Judging metrics on the held-out fold 0

final-A was trained on all labels, so it cannot be scored offline. The held-out numbers come from its
**fold-0 twin E9**, which has the same architecture, objective and self-training recipe but was trained
without the fold-0 labels. Fold 0 has 120 observations, 192 annotator records and 1,341 filaments. The
metrics follow the organizers' self-evaluation notebook (`scripts/rubric_metrics.py`):
- PQ per annotator record, pooled over records;
- IoU and Dice of every overlapping ground-truth/prediction pair;
- the one-to-many and many-to-one relations;
- the `torchmetrics` DiceScore of the binary filament mask.

With `RUN_FOLD0 = False` this section shows the released results in `docs/`. With `RUN_FOLD0 = True` it
recomputes them from `FOLD0_CHECKPOINT`.

In [ ]:
if RUN_FOLD0:
    assert FOLD0_CHECKPOINT.exists(), "train configs/v9_e9.yaml (section 9) or place a fold-0 model at FOLD0_CHECKPOINT"
    run(sys.executable, "-m", "scripts.dump_tta_candidates", "--checkpoint", FOLD0_CHECKPOINT, "--split", "val",
        "--output", FOLD0_DUMP)
    V = ["--folders", FOLD0_DUMP, "--folder-views", VIEWS, "--votes", "1", "--score", ".10", "--mask", ".40"]
    R = {"rubric": OUT / "fold0/rubric", "morphology": OUT / "fold0/morphology", "shape": OUT / "fold0/shape",
         "details": OUT / "fold0/morphology_details.png", "errors": OUT / "fold0/error_modes.json"}
    run(sys.executable, "-m", "scripts.rubric_metrics", *V, "--workers", WORKERS, "--output-dir", R["rubric"])
    run(sys.executable, "-m", "scripts.morphology_figures", *V, "--per-record", R["rubric"] / "per_record.json",
        "--output-dir", R["morphology"])
    run(sys.executable, "-m", "scripts.shape_statistics", *V, "--csv", SUBMISSION, "--workers", WORKERS,
        "--output-dir", R["shape"])
    run(sys.executable, "-m", "scripts.morphology_details", *V, "--workers", WORKERS, "--output", R["details"])
    run(sys.executable, "-m", "scripts.error_modes", *V, "--workers", WORKERS, "--output", R["errors"])
else:
    R = {"rubric": REPO / "docs/figures/rubric_fold0", "morphology": REPO / "docs/figures/morphology",
         "shape": REPO / "docs/figures/shape", "details": REPO / "docs/figures/morphology_details.png",
         "errors": REPO / "docs/analysis/error_modes_fold0.json"}

m = json.loads((R["rubric"] / "rubric_metrics.json").read_text())
ci = m.get("bootstrap", {}).get("ci95", {})   # 95 % intervals over resampled observations
interval = lambda key: f" [{ci[key][0]:.3f}, {ci[key][1]:.3f}]" if key in ci else ""
gt, pred = m["one_to_many_gt_degree"], m["many_to_one_pred_degree"]
degrees = lambda d: " / ".join(f"{d.get(str(k), 0):,}" for k in range(4))
display(Markdown(f"""
| Fold 0, 8 views, submission setting | value |
|---|---|
| PQ (SQ / RQ) [95 % CI] | **{m['pq']:.3f}** ({m['sq']:.3f} / {m['rq']:.3f}){interval('pq')} |
| TP / FP / FN | {m['tp']} / {m['fp']} / {m['fn']} |
| Dice of overlapping pairs: mean (median) [95 % CI] | {m['pair_dice_mean']:.3f} ({m['pair_dice_quantiles_10_25_50_75_90'][2]:.3f}){interval('pair_dice_mean')} |
| IoU of overlapping pairs: mean (median) | {m['pair_iou_mean']:.3f} ({m['pair_iou_quantiles_10_25_50_75_90'][2]:.3f}) |
| torchmetrics DiceScore per record: mean / median / pooled | {m['semantic_dice_mean']:.3f} / {m['semantic_dice_median']:.3f} / {m['semantic_dice_global']:.3f} |
| one-to-many: GT overlapped by 0 / 1 / 2 / 3 predictions | {degrees(gt)} |
| many-to-one: prediction overlapping 0 / 1 / 2 / 3 GT | {degrees(pred)} |
"""))
display(Picture(filename=str(R["rubric"] / "rubric_distributions.png"), width=1000))

**Morphology on fold 0.** The figures show, in order:
- the best, median and hardest observations by semantic Dice, with every annotator record;
- shape descriptors of the annotations, the fold-0 predictions and the test predictions;
- close-ups chosen at fixed ranks, not by hand: matched filaments at the 20/40/60/80th IoU percentile,
  predictions made of several parts, and missed filaments under 400 px.

Annotator outlines are yellow and predictions cyan.

In [ ]:
for png in sorted(R["morphology"].glob("*.png")):
    display(Picture(filename=str(png), width=900))
display(Picture(filename=str(R["shape"] / "shape_statistics.png"), width=1000))
display(Picture(filename=str(R["details"]), width=1000))
c = json.loads(R["errors"].read_text())["counts"]
size = lambda k: f"{c[k + '_area_small']} / {c[k + '_area_medium']} / {c[k + '_area_large']}"
display(Markdown(f"""
| Where PQ is lost (fold 0) | count | area < 800 / 800–3,000 / > 3,000 px |
|---|---|---|
| missed, no prediction overlaps it (IoU ≤ 0.1) | {c['fn_no_candidate']} | {size('fn_no_candidate')} |
| missed, near miss (best IoU 0.1–0.5: extent, split or merge) | {c['fn_near_miss']} | {size('fn_near_miss')} |
| missed, correct candidate with a low score | {c['fn_low_score']} | {size('fn_low_score')} |
| false positive, near miss of a drawn filament | {c['fp_near_miss']} | |
| false positive, nothing drawn there in that record | {c['fp_unlabeled']} | |
"""))

## 8. Why the score saturates: label visibility (analysis)

Filaments are absorption features. Their visibility in H-alpha is set by the cloud-model contrast
$C = I/B - 1$ inside the outline. `research/label_visibility.py` measures two things against $C$, size and
disk position:
- how often the *other* annotators of an observation also drew a filament (IoU > 0.5);
- how often the model matched or missed it on fold 0.

Results:
- **Labels follow a visibility curve.** Annotator agreement falls from 0.79 for the darkest filaments to 0.25
  for the faintest.
- **The model's misses follow the same curve.**
- **The model agrees with the annotators better than they agree with each other.** On the same multi-record
  fold-0 observations, the model reaches PQ 0.43 against the records, and human against human reaches 0.33.

`research/magnetogram_pil.py` adds a physical check. It pairs the training observations with GONG and SDO/HMI
line-of-sight magnetograms; these are used for analysis only, never for training or inference. Annotated
filaments lie within 20 Mm of a polarity inversion line 4.8× more often than random disk points. The model's
unsupported detections are as close to these lines as matched filaments, so they look like real filaments that
an annotator skipped.

In [ ]:
if RUN_RESEARCH:
    assert FOLD0_DUMP.exists(), "run section 7 with RUN_FOLD0 = True first"
    run(sys.executable, "research/label_visibility.py", "--folder", FOLD0_DUMP,
        "--output", OUT / "label_visibility.json", "--figure", OUT / "visibility")
    if (REPO / "data/gong_bqa/fits").is_dir() and (REPO / "data/hmi_720s/fits").is_dir():
        run(sys.executable, "research/magnetogram_pil.py", "--folder", FOLD0_DUMP, "--output", OUT / "magnetogram_pil.json")
    V_JSON, V_FIG, M_JSON = OUT / "label_visibility.json", OUT / "visibility.png", OUT / "magnetogram_pil.json"
else:
    V_JSON, V_FIG, M_JSON = (REPO / "research/label_visibility.json", REPO / "docs/figures/visibility.png",
                             REPO / "research/magnetogram_pil.json")

v = json.loads(V_JSON.read_text())
rows = "\n".join(f"| {h[0]} | {h[1]} | {h[2]:.2f} | {s[2][0]:.2f} | {s[2][2]:.2f} |"
                 for h, s in zip(v["human_agreement_vs_contrast"], v["model_status_vs_contrast"]))
sigma_rows = "\n".join(f"| {h[0]} | {h[1]} | {h[2]:.2f} | {s[2][0]:.2f} | {s[2][2]:.2f} |" for h, s in
                       zip(v["human_agreement_vs_contrast_sigma"], v["model_status_vs_contrast_sigma"]))
model, human = v["model_vs_fold0_records_multi_record_observations"], v["human_vs_human_fold0_multi_record_observations"]
boot = v.get("model_vs_human_bootstrap")   # paired bootstrap over the observations
gap = (f"; difference 95 % CI [{boot['difference_ci95'][0]:.3f}, {boot['difference_ci95'][1]:.3f}]"
       if boot else "")
display(Markdown(f"""
| contrast $C$ | annotated filaments | annotator agreement | model matched (fold 0) | model missed (fold 0) |
|---|---|---|---|---|
{rows}

| $-C$ in units of the quiet-Sun texture $\\sigma$ (median $\\sigma$ = {v['quiet_sun_texture_sigma_p10_p50_p90'][1]:.3f}) | annotated filaments | annotator agreement | model matched | model missed |
|---|---|---|---|---|
{sigma_rows}

Same multi-record fold-0 observations: model vs records PQ **{model['pq@0.5']:.3f}**, human vs human PQ **{human['pq@0.5']:.3f}**{gap}.
"""))
display(Picture(filename=str(V_FIG), width=600))
if M_JSON.exists():
    g = json.loads(M_JSON.read_text())
    share = lambda key: f"{100 * g[key]['within_20mm']:.0f} %"
    display(Markdown(f"""
| within 20 Mm of a polarity inversion line | share |
|---|---|
| random disk points | {share('ALL_null_random_disk_points')} |
| annotated filaments ({g['observations']} observations) | {share('ALL_annotated_filaments')} |
| fold 0: matched predictions / unsupported predictions / missed filaments | {share('fold0_pred_matched')} / {share('fold0_pred_unsupported')} / {share('fold0_gt_missed')} |
| disk centre (μ ≥ 0.85): filaments / random points | {share('mu_0.85_1.01_annotated')} / {share('mu_0.85_1.01_random')} |
| near the limb (μ < 0.6): filaments / random points | {share('mu_0_0.6_annotated')} / {share('mu_0_0.6_random')} |
"""))

## 9. Training chain (several GPU-days; off by default)

Every stage is a small YAML change of its predecessor (`configs/`, `base:` inheritance) and warm-starts from
its weights (`--init-from`). The supervised stages S1–S4 and the teachers use the fold-0 training split,
with fold 0 for model selection.

| Stage | Config | Change | Fold-0 PQ, single view |
|---|---|---|---|
| S1 | `dual_3090` | ImageNet FlashInternImage-L, query decoder | 0.169 |
| S2 | `pq_aligned_v1` | differentiable PQ surrogate as the main loss | 0.348 |
| S3 | `pq_seeded_v3` | mask pilots, image-seeded queries, balanced native sampling | 0.375 |
| S4 = v7 | `pq_recovery_v7` | soft surrogate, log objective | 0.395 |
| E3 / E6 | `v9_e3` / `v9_e6` | limb-flattened channel, rotation/scale augmentation (E6: multi-scale views) | 0.403 / 0.400 |
| E9 | `v9_e9` | E3 + round-1 pseudo-labels (fold-0 twin, round-2 teacher) | 0.413 |
| F1 | `v9_f1` | E9 recipe on all labels | – |
| **final-A** | `v9_ffinal_a` | F1 continued on the round-2 pool, 0.25× learning rate, 12 epochs | – |

**Noisy-Student self-training.**
- Unlabelled GONG archive frames from the same instrument (`scripts/download_gong.py`) are pseudo-labelled by
  two fold-0 teachers, using views d0 and f2 of each model, fused like a submission with mask threshold 0.70
  (`scripts/pseudo_label.py`).
- Frames from test-image days are always excluded. The recipe below also excludes the fold-0 validation days,
  so E9's fold-0 score stays clean. The original E9 pool did not exclude them (report §6).
- Each epoch adds as many pseudo-labelled frames as there are labelled records.
- The submitted model's round-2 pool held 9,272 frames from 3,465 days. Archive availability makes pools
  differ slightly between downloads; more random-day batches (`--selection-file`) enlarge the pool.

In [ ]:
if RUN_TRAINING:
    import yaml
    train = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={GPUS}",
             "-m", "solar_filaments.train"]

    def stage(config, init=None):
        run(*train, "--config", config, *(["--init-from", init] if init else []))

    def with_pools(name, *pools):
        """configs/<name>.yaml trained on other pseudo-label pools (written under OUT/configs)."""
        path = OUT / "configs" / f"{name}.yaml"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(yaml.safe_dump({"base": f"configs/{name}.yaml", "train": {"extra_data_roots": list(pools)}}))
        return path

    def pseudo_label(teachers, pool):
        Path(pool).mkdir(parents=True, exist_ok=True)
        (Path(pool) / "exclude_fold.txt").write_text("0")   # test-image days are always excluded
        run(sys.executable, "-m", "scripts.pseudo_label", "--images", "data/gong_extra/images", "--checkpoints", *teachers,
            "--views", "d0", "f2", "--votes", "1", "--score", ".10", "--mask", ".70", "--output", pool)
        run(sys.executable, "-m", "scripts.pseudo_label", "assemble", pool, "data/gong_extra/images")

    run(sys.executable, "scripts/download_weights.py")                           # ImageNet-22k->1k FlashInternImage-L
    run(sys.executable, "scripts/prepare_data.py", "--config", "configs/v9_ffinal_a.yaml")  # cached geometry targets
    stage("configs/dual_3090.yaml")                                              # S1
    stage("configs/pq_aligned_v1.yaml", "runs/dual_3090/best.pt")                # S2
    stage("configs/pq_seeded_v3.yaml", "runs/pq_aligned_v1/best.pt")             # S3
    stage("configs/pq_recovery_v7.yaml", "runs/pq_seeded_v3/best.pt")            # S4 = v7
    stage("configs/v9_e3.yaml", "runs/pq_recovery_v7/best.pt")                   # teacher E3
    stage("configs/v9_e6.yaml", "runs/pq_recovery_v7/best.pt")                   # teacher E6

    run(sys.executable, "scripts/download_gong.py", "--output", "data/gong_extra")   # competition days + random days
    run(sys.executable, "scripts/download_gong.py", "--output", "data/gong_extra", "--per-competition-day", "0",
        "--random-days", "1000", "--per-random-day", "3", "--seed", "2028", "--selection-file", "selection_batch2.json")

    pseudo_label(["runs/v9_e3/best.pt", "runs/v9_e6/best.pt"], "data/gong_pseudo_r1")         # round 1
    stage(with_pools("v9_e9", "data/gong_pseudo_r1"), "runs/pq_recovery_v7/best.pt")         # E9 (fold-0 twin)
    stage(with_pools("v9_f1", "data/gong_pseudo_r1"), "runs/pq_recovery_v7/best.pt")         # F1 (all labels)
    pseudo_label(["runs/v9_e9/best.pt", "runs/v9_e6/best.pt"], "data/gong_pseudo_r2")         # round 2
    stage("configs/v9_ffinal_a.yaml", "runs/v9_f1/final.pt")                                  # final-A
    print("final-A weights: runs/v9_ffinal_a/final.pt; set WEIGHTS to it and rerun section 5")

## Summary

| | |
|---|---|
| Submitted model | final-A: FlashInternImage-L query model (223.6 M parameters) + 8-view dihedral TTA |
| Public leaderboard | PQ 0.40 |
| Fold 0 (fold-0 twin E9) | PQ 0.446 · Dice of overlapping pairs 0.728 · DiceScore 0.656 per record |
| Human vs human (training set) | PQ 0.340 |
| Inference cost | ~3.7 s per test image on one RTX 3090 (8 views) |

Further reading:
- `docs/TECHNICAL_REPORT.md`: the method, ablations and negative results, compliance, reproduction.
- `paper/`: the 4-page report.
- `README.md`: the same commands for the command line.